In [ ]:
import socket
import threading
import json
import os
import re
import hashlib
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.llms import Ollama
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain.chains import create_retrieval_chain
import traceback

class ChatbotServer:
    def __init__(self, host='127.0.0.1', port=65432):
        self.host = host
        self.port = port
        self.retrieval_chain = None
        self.current_pdf_path = None
        self.lock = threading.Lock()
        self.server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self.server_socket.bind((self.host, self.port))
        print(f"Server listening on {self.host}:{self.port}")

    def get_document_signature(self, file_path):
        """Generates a hash of the PDF file's content."""
        hasher = hashlib.sha256()
        try:
            with open(file_path, 'rb') as f:
                while chunk := f.read(4096):
                    hasher.update(chunk)
            return hasher.hexdigest()
        except Exception:
            return None

    def initialize_chatbot_logic(self, file_path):
        with self.lock:
            self.current_pdf_path = file_path
            try:
                persist_directory = "./chroma_db"
                os.makedirs(persist_directory, exist_ok=True)
                signature_file = os.path.join(persist_directory, "doc_signature.txt")
                current_signature = self.get_document_signature(self.current_pdf_path)

                load_from_disk = False
                if os.path.exists(signature_file):
                    with open(signature_file, 'r') as f:
                        saved_signature = f.read()
                    if saved_signature == current_signature:
                        load_from_disk = True

                if load_from_disk:
                    print("Loading existing embeddings...")
                    embeddings = OllamaEmbeddings(model="nomic-embed-text")
                    vector_store = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
                else:
                    print("Creating new embeddings...")
                    loader = PyPDFLoader(self.current_pdf_path)
                    docs = loader.load()
                    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
                    all_splits = text_splitter.split_documents(docs)
                    embeddings = OllamaEmbeddings(model="nomic-embed-text")
                    vector_store = Chroma.from_documents(documents=all_splits, embedding=embeddings, persist_directory=persist_directory)
                    with open(signature_file, 'w') as f:
                        f.write(current_signature)

                llm = Ollama(model="gemma3:1b")
                retriever = vector_store.as_retriever()
                prompt = ChatPromptTemplate.from_template("""Answer the user's question based on the provided context:
<context>
{context}
</context>
Question: {input}""")
                document_chain = create_stuff_documents_chain(llm, prompt)
                self.retrieval_chain = create_retrieval_chain(retriever, document_chain)
                
                return {"status": "success", "message": "Backend ready."}

            except Exception as e:
                traceback.print_exc()
                return {"status": "error", "message": f"Failed to initialize backend. Error: {e}"}

    def get_response(self, user_text):
        with self.lock:
            if not self.retrieval_chain:
                return {"status": "error", "message": "Please select a document first."}
            
            try:
                response = self.retrieval_chain.invoke({"input": user_text})
                chatbot_response = response['answer']
                
                return {"status": "success", "response": chatbot_response}
            except Exception as e:
                traceback.print_exc()
                return {"status": "error", "message": f"An error occurred: {e}"}

    def handle_client_request(self, conn, addr):
        print(f"Connected by {addr}")
        while True:
            try:
                data = conn.recv(4096)
                if not data:
                    break
                
                request = json.loads(data.decode('utf-8'))
                
                if request['type'] == 'init_pdf':
                    print(f"Received init_pdf request for: {request['file_path']}")
                    response = self.initialize_chatbot_logic(request['file_path'])
                    conn.sendall(json.dumps(response).encode('utf-8'))
                elif request['type'] == 'query':
                    print(f"Received query request: {request['text']}")
                    response = self.get_response(request['text'])
                    conn.sendall(json.dumps(response).encode('utf-8'))
                
            except json.JSONDecodeError:
                print("Invalid JSON received.")
                break
            except BrokenPipeError:
                print("Client disconnected.")
                break
            except Exception as e:
                print(f"Error handling client request: {e}")
                traceback.print_exc()
                break

        conn.close()
        print(f"Connection with {addr} closed.")

    def start(self):
        self.server_socket.listen()
        while True:
            conn, addr = self.server_socket.accept()
            client_thread = threading.Thread(target=self.handle_client_request, args=(conn, addr))
            client_thread.daemon = True
            client_thread.start()

if __name__ == "__main__":
    server = ChatbotServer()
    server.start()

Server listening on 127.0.0.1:65432
Connected by ('127.0.0.1', 49710)
Received init_pdf request for: /home/gururaj/Downloads/Data science.pdf
Creating new embeddings...


/tmp/ipykernel_35432/521518224.py:65: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")
/tmp/ipykernel_35432/521518224.py:70: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="gemma3:1b")


Received query request: summarize datascience
